# 03 · EDA temporal

**Fase CRISP-DM:** comprensión de los datos.

**Objetivo:** decidir con evidencia cómo modelar la serie mensual por segmento: qué rezagos usar, cómo representar la estacionalidad, con qué ventana evaluar y si conviene un modelo conjunto o uno por segmento.

**Regla de privacidad:** solo usa la serie agregada por mes y segmento, leída del lakehouse.

In [ ]:
import warnings

import exp_a
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, adfuller, pacf

from lojagas.ml.walk_forward import particiones

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
exp_a.estilo_graficos()
warnings.filterwarnings("ignore", category=FutureWarning, message="adfuller")

serie = exp_a.cargar_serie()
y = serie.pivot_table(index="mes", columns="segmento", values="toneladas").asfreq("MS")
SEGMENTOS = list(y.columns)
H = exp_a.HORIZONTE
assert y.notna().all().all(), "hay meses sin dato"
print(f"{len(y)} meses, de {y.index[0]:%Y-%m} a {y.index[-1]:%Y-%m} | segmentos: {SEGMENTOS}")


def eje_anual(eje):
    eje.xaxis.set_major_locator(mdates.YearLocator())
    eje.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

## 1. Nivel y crecimiento

El último año está incompleto, así que los años se comparan sobre los mismos meses.

In [ ]:
ultimo_mes = y.index[-1].month
comparables = y[y.index.month <= ultimo_mes]
acumulado = comparables.groupby(comparables.index.year).sum()
print(f"Toneladas de enero al mes {ultimo_mes} de cada año")
display(acumulado.round(0))
print("Variación % frente al mismo período del año anterior")
display(acumulado.pct_change().dropna().mul(100).round(1))
print("Participación % en el volumen total")
display((y.sum() / y.sum().sum() * 100).round(2).to_frame("participacion_%").T)

## 2. Tendencia, estacionalidad y residuo (STL)

Con menos de cuatro ciclos anuales, la estacionalidad se asume fija (ventana estacional "periódica"): con la ventana por defecto, la descomposición sobreajusta el patrón de cada mes y deja el residuo casi en cero.

La fuerza de cada componente va de 0 a 1 (Hyndman): cerca de 1, el componente domina sobre el residuo. El residuo como % del nivel es el ruido que ningún patrón regular explica: una cota de lo que un modelo puede mejorar.

In [ ]:
# Ventana muy ancha y grado 0: el índice de cada mes es el mismo todos los años
PERIODICA = {"seasonal": 10 * len(y) + 1, "seasonal_deg": 0}
stl = {s: STL(y[s], period=12, robust=True, **PERIODICA).fit() for s in SEGMENTOS}


def fuerza(componente, resto):
    """1 − Var(resto) / Var(componente + resto), acotada en 0."""
    return max(0.0, 1 - resto.var() / (componente + resto).var())


componentes = pd.DataFrame(
    {
        s: {
            "fuerza_tendencia": fuerza(r.trend, r.resid),
            "fuerza_estacionalidad": fuerza(r.seasonal, r.resid),
            "amplitud_estacional_%": (r.seasonal.max() - r.seasonal.min()) / y[s].mean() * 100,
            "residuo_%_del_nivel": r.resid.std() / y[s].mean() * 100,
            "cv_serie_%": y[s].std() / y[s].mean() * 100,
        }
        for s, r in stl.items()
    }
).T
display(componentes.round(2))

In [ ]:
fig, ejes = plt.subplots(3, len(SEGMENTOS), figsize=(4.3 * len(SEGMENTOS), 6.8), sharex=True)
for j, s in enumerate(SEGMENTOS):
    r, color = stl[s], exp_a.color_segmento(s, SEGMENTOS)
    ejes[0, j].plot(y.index, y[s], color=color, label="observado")
    ejes[0, j].plot(y.index, r.trend, color=exp_a.TINTA, label="tendencia")
    ejes[0, j].set_title(f"{s} · toneladas")
    ejes[1, j].plot(y.index, r.seasonal, color=color)
    ejes[1, j].set_title("componente estacional (t)")
    ejes[2, j].bar(y.index, r.resid, width=18, color=color)
    ejes[2, j].set_title("residuo (t)")
    eje_anual(ejes[2, j])
ejes[0, 0].legend(loc="best")
plt.tight_layout()
plt.show()

## 3. Perfil estacional

Índice estacional de cada mes del año, en % del nivel medio del segmento. La tabla de armónicos dice cuántos pares seno/coseno hacen falta para reproducir ese perfil: si un solo armónico explica poco, la codificación `mes_sin`/`mes_cos` no alcanza.

In [ ]:
perfil = pd.DataFrame(
    {
        s: r.seasonal.groupby(r.seasonal.index.month).mean() / y[s].mean() * 100
        for s, r in stl.items()
    }
)
perfil.index.name = "mes_del_año"
display(perfil.round(1))
display(
    pd.DataFrame(
        {
            "mes_pico": perfil.idxmax(),
            "pico_%": perfil.max(),
            "mes_valle": perfil.idxmin(),
            "valle_%": perfil.min(),
        }
    ).round(1)
)


def r2_armonicos(p, k):
    """R² de ajustar el perfil estacional con k armónicos anuales."""
    m, valores = p.index.to_numpy(), p.to_numpy()
    senales = [f(2 * np.pi * j * m / 12) for j in range(1, k + 1) for f in (np.sin, np.cos)]
    X = np.column_stack([np.ones(len(m)), *senales])
    ajuste = X @ np.linalg.lstsq(X, valores, rcond=None)[0]
    return 1 - ((valores - ajuste) ** 2).sum() / ((valores - valores.mean()) ** 2).sum()


display(
    pd.DataFrame(
        {
            f"R2_con_{k}_armonicos": {s: r2_armonicos(perfil[s], k) for s in SEGMENTOS}
            for k in (1, 2, 3)
        }
    ).round(2)
)

In [ ]:
fig, eje = plt.subplots(figsize=(8, 3.6))
for s in SEGMENTOS:
    eje.plot(
        perfil.index, perfil[s], color=exp_a.color_segmento(s, SEGMENTOS), label=s, marker="o", ms=5
    )
eje.axhline(0, color=exp_a.EJE, lw=1)
eje.set_xticks(range(1, 13))
eje.set_xlabel("mes del año")
eje.set_title("Índice estacional (% del nivel medio del segmento)")
eje.legend(loc="best")
plt.tight_layout()
plt.show()

## 4. ¿Cuánto del patrón es calendario? Días de venta

Los meses no tienen los mismos días ni la misma mezcla de días de la semana. Los "días de venta" de un mes son sus días ponderados por la actividad de cada día de la semana (medida solo con el primer año). Es calendario puro: se conoce de antemano para cualquier mes futuro.

Dos pruebas de que importa: la correlación entre el perfil estacional y los días de venta de cada mes del año, y la correlación entre el cambio interanual de las toneladas y el de los días de venta. La segunda es la que dice si ajustar el estacional ingenuo por días lo mejoraría.

In [ ]:
pesos = exp_a.cargar_pesos_dias()
if pesos is not None:
    print("Actividad relativa por día de la semana (1 = día promedio)")
    display(
        pesos.pivot_table(index="segmento", columns="dia_semana", values="peso")
        .set_axis(["lun", "mar", "mié", "jue", "vie", "sáb", "dom"], axis=1)
        .round(2)
    )
dias = serie.pivot_table(index="mes", columns="segmento", values="dias_equiv").asfreq("MS")
desvio_dias = dias.groupby(dias.index.month).mean() / dias.mean() * 100 - 100
display(
    pd.DataFrame(
        {
            "dias_de_venta_min": dias.min(),
            "dias_de_venta_max": dias.max(),
            "variacion_interanual_max_%": dias.pct_change(12).abs().max() * 100,
            "corr_perfil_estacional_vs_dias": perfil.corrwith(desvio_dias),
            "corr_cambio_interanual_t_vs_dias": y.pct_change(12).corrwith(dias.pct_change(12)),
        }
    ).round(2)
)

## 5. Autocorrelación: qué rezagos aportan

Con horizonte 2, el rezago 1 no existe al predecir: solo sirven los rezagos de 2 meses o más. La primera tabla es la autocorrelación de la serie. La segunda es la de la serie con diferencia estacional (el mes menos el mismo mes del año anterior): dice si, una vez descontado lo que ya explica el rezago 12, los meses recientes todavía aportan.

In [ ]:
K = 14
cota = 1.96 / np.sqrt(len(y))
indice = pd.RangeIndex(1, K + 1, name="rezago")
acf_serie = pd.DataFrame({s: acf(y[s], nlags=K)[1:] for s in SEGMENTOS}, index=indice)
pacf_serie = pd.DataFrame(
    {s: pacf(y[s], nlags=K, method="ywm")[1:] for s in SEGMENTOS}, index=indice
)
print(f"Autocorrelación de la serie (significativa si |valor| > {cota:.2f})")
display(acf_serie.round(2).T)
print("Autocorrelación parcial")
display(pacf_serie.round(2).T)

dif = y.diff(12).dropna()
cota_dif = 1.96 / np.sqrt(len(dif))
acf_dif = pd.DataFrame(
    {s: acf(dif[s], nlags=8)[1:] for s in SEGMENTOS}, index=pd.RangeIndex(1, 9, name="rezago")
)
print(f"Autocorrelación tras la diferencia estacional (significativa si |valor| > {cota_dif:.2f})")
display(acf_dif.round(2).T)

In [ ]:
fig, ejes = plt.subplots(2, len(SEGMENTOS), figsize=(4.3 * len(SEGMENTOS), 5.2), sharey="row")
for j, s in enumerate(SEGMENTOS):
    color = exp_a.color_segmento(s, SEGMENTOS)
    for fila, (tabla, limite, titulo) in enumerate(
        [(acf_serie, cota, "serie"), (acf_dif, cota_dif, "dif. estacional")]
    ):
        eje = ejes[fila, j]
        # En gris, los rezagos que no existen al predecir con este horizonte
        colores = [exp_a.TENUE if k < H else color for k in tabla.index]
        eje.bar(tabla.index, tabla[s], width=0.55, color=colores)
        eje.axhline(0, color=exp_a.EJE, lw=1)
        for signo in (1, -1):
            eje.axhline(signo * limite, color=exp_a.TINTA_SECUNDARIA, lw=1)
        eje.set_xticks(list(tabla.index))
        eje.set_title(f"{s} · ACF de la {titulo}")
fig.supxlabel(
    f"rezago en meses · en gris, los que no existen al predecir con horizonte {H}",
    fontsize=9,
    color=exp_a.TINTA_SECUNDARIA,
)
plt.tight_layout()
plt.show()

## 6. Estacionariedad (ADF)

Con unas pocas decenas de observaciones la prueba tiene poco poder: sirve como indicio, no como veredicto. Un p alto en la serie y bajo en la diferencia estacional apunta a que modelar en relación con el año anterior es más estable que modelar el nivel.

In [ ]:
display(
    pd.DataFrame(
        {
            s: {
                "p_serie": adfuller(y[s], autolag="AIC")[1],
                "p_diferencia_estacional": adfuller(dif[s], autolag="AIC")[1],
            }
            for s in SEGMENTOS
        }
    ).T.round(3)
)

## 7. Meses atípicos

Residuo de la descomposición robusta a más de 3 desviaciones robustas (MAD). Son meses que ningún patrón regular explica; conviene preguntar a Lojagas qué pasó.

In [ ]:
filas = []
for s, r in stl.items():
    centro = r.resid.median()
    mad = (r.resid - centro).abs().median() * 1.4826
    z = (r.resid - centro) / mad
    for mes, valor in z[z.abs() > 3].items():
        filas.append(
            {
                "segmento": s,
                "mes": f"{mes:%Y-%m}",
                "z_robusto": valor,
                "desvio_%_del_nivel": r.resid[mes] / y[s].mean() * 100,
            }
        )
atipicos = pd.DataFrame(filas, columns=["segmento", "mes", "z_robusto", "desvio_%_del_nivel"])
print(f"meses atípicos: {len(atipicos)}")
display(atipicos.round(1))

## 8. Clientes activos frente a volumen

¿El número de clientes que compran anticipa el volumen? Lo que importa para el modelo es la última columna: la correlación entre el crecimiento interanual de las toneladas y el de los clientes de dos meses antes, que es el dato disponible al predecir.

In [ ]:
if "clientes_activos" in serie.columns:
    c = serie.pivot_table(index="mes", columns="segmento", values="clientes_activos").asfreq("MS")
    crece_y, crece_c = y.pct_change(12), c.pct_change(12)
    display(
        pd.DataFrame(
            {
                "clientes_primer_año": c.iloc[:12].mean(),
                "clientes_último_año": c.iloc[-12:].mean(),
                "clientes_cambio_%": (c.iloc[-12:].mean() / c.iloc[:12].mean() - 1) * 100,
                "toneladas_cambio_%": (y.iloc[-12:].mean() / y.iloc[:12].mean() - 1) * 100,
                "t_por_cliente_primer_año": y.iloc[:12].sum() / c.iloc[:12].sum(),
                "t_por_cliente_último_año": y.iloc[-12:].sum() / c.iloc[-12:].sum(),
                "corr_crecimientos_mismo_mes": crece_y.corrwith(crece_c),
                f"corr_con_clientes_{H}_meses_antes": crece_y.corrwith(crece_c.shift(H)),
            }
        ).round(2)
    )
else:
    print("El lakehouse no tiene la tabla de clientes activos")

## 9. ¿Los segmentos se mueven juntos?

Correlación entre los residuos de los segmentos. Si es baja, lo que sorprende a un segmento no dice nada del otro: un modelo conjunto solo ayuda por compartir la forma de la relación, no la información del mes.

In [ ]:
display(pd.DataFrame({s: r.resid for s, r in stl.items()}).corr().round(2))

## 10. Diseño de la evaluación: cuántas particiones caben

El rezago 12 y el nivel del último año cerrado necesitan 13 meses previos, así que no todos los meses son evaluables. La ventana mínima reparte lo que queda entre entrenar y probar. Una evaluación que no cubre los 12 meses del año queda sesgada por la estación.

In [ ]:
datos = exp_a.tabla_modelado(serie)
print(
    f"meses evaluables: {datos['mes'].nunique()} "
    f"(de {datos['mes'].min():%Y-%m} a {datos['mes'].max():%Y-%m})"
)
filas = []
for ventana in (12, 18, 24):
    pruebas = [
        datos["mes"].iloc[te].iloc[0] for _, te in particiones(datos["mes"], ventana, horizonte=H)
    ]
    filas.append(
        {
            "ventana_minima": ventana,
            "particiones": len(pruebas),
            "primera_prueba": f"{pruebas[0]:%Y-%m}",
            "ultima_prueba": f"{pruebas[-1]:%Y-%m}",
            "meses_del_año_cubiertos": len({p.month for p in pruebas}),
        }
    )
display(pd.DataFrame(filas).set_index("ventana_minima"))

## Decisiones

Las cifras quedan en las salidas de este notebook, que no se suben al repositorio.

1. **Los segmentos son procesos distintos.** El doméstico tiene tendencia y un patrón anual suave; el agroindustrial, un ciclo semestral muy marcado y mucho ruido; el industrial casi no tiene estacionalidad. Sus residuos apenas se correlacionan. Se espera que un modelo por segmento funcione mejor que uno conjunto; se prueban los dos.
2. **Estacionalidad fija y con dos armónicos.** Con menos de cuatro ciclos no se puede estimar un patrón que cambie. Un solo par seno/coseno no reproduce el ciclo semestral del agroindustrial: se agregan `mes_sin2` y `mes_cos2`.
3. **Días de venta.** Casi no se vende los domingos y el sábado es medio día, así que la mezcla de días de la semana mueve el volumen del mes. En el doméstico, el cambio interanual de los días de venta explica buena parte del cambio interanual de las toneladas. Se agrega la variable `dias_equiv` y una referencia nueva sin aprendizaje: el estacional ingenuo ajustado por días de venta. Los feriados no están incluidos (son de la configuración B).
4. **Rezagos 2, 3 y 12.** El 4 no es significativo en dos de los tres segmentos. El 6 solo importa en el agroindustrial, donde ya lo cubre el segundo armónico: se prueba aparte en el notebook 06.
5. **Escala relativa.** El nivel cambia con la tendencia y los segmentos difieren en dos órdenes de magnitud. Dividir por el nivel del último año cerrado los vuelve comparables.
6. **Clientes activos: fuera de las variables base.** Su crecimiento no anticipa el del volumen; en el doméstico crecen mucho más que las toneladas, lo que apunta a un cambio en cómo se registran los clientes. Se prueba aparte.
7. **Evaluación con ventana mínima de 12 meses.** Es la única que da una veintena de particiones y cubre los 12 meses del año. Con 24 meses las pruebas cubren solo una parte del año y la comparación queda sesgada por la estación. Los resultados se reportan también para las particiones con 18 meses o más de historia.
8. **Meses atípicos.** Hay que preguntar a Lojagas qué pasó en ellos antes de decidir si se corrigen o se dejan.